# 0825_dongjin_024_xgb_native_thresholds

This notebook directly extracts the mathematical split thresholds learned by the XGBoost baseline (`0824_kimjaehak_005_xgboost_baseline.pkl`). 
It parses the native XGBoost internal trees (`trees_to_dataframe()`) to find exactly what boundary conditions trigger False Positives (False Calls).
Crucially, as per the user's request, this is analyzed **by inspection_type** to provide type-specific rules.

In [1]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

EXPERIMENT_ID = "0825_dongjin_024_xgb_native_thresholds"
MODEL_PATH = Path("../models/0824_kimjaehak_005_xgboost_baseline.pkl")
DATA_PATH = Path("../data/raw/dataset.csv")

TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5

E:\pro_newton\제조 AI\팀 과제\siemens_aoi_ML_practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data and Reproduce Test Set Split

In [2]:
# 1. Load Data
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
if raw_df.columns[0].startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={raw_df.columns[0]: RECORD_ID})

# 2. Deduplicate
dedup_columns = [col for col in raw_df.columns if col not in {RECORD_ID, TIME_COLUMN}]
clean_df = raw_df.loc[~raw_df.duplicated(subset=dedup_columns, keep="first")].copy().reset_index(drop=True)

# 3. Prepare inputs
clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
feature_columns = [col for col in clean_df.columns if col not in {RECORD_ID, TIME_COLUMN, TARGET}]

X = clean_df[feature_columns].copy()
y = clean_df[TARGET].astype("int8").copy()
timestamps = clean_df[TIME_COLUMN].copy()

# 4. Time-based split (60/20/20) as in baseline 005
timestamp_group_sizes = timestamps.value_counts(sort=False).sort_index()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
valid_end_position = int(np.searchsorted(cumulative_rows, len(clean_df) * 0.80, side="left"))
valid_end_time = timestamp_group_sizes.index[valid_end_position]

test_mask = timestamps > valid_end_time
X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

print(f"Test Set Size: {len(X_test)}")


Test Set Size: 78396


## 2. Load Model and Identify False Calls

In [3]:
artifact = joblib.load(MODEL_PATH)
model = artifact['model']

# Note: The model might have been trained on data WITHOUT 'inspection_type' if it was dropped,
# or it might have been kept. We will pass X_test as is, if it fails, we will align columns.
try:
    test_probabilities = model.predict_proba(X_test)[:, 1]
except ValueError:
    # If there's a column mismatch, we use only the columns the model expects
    test_probabilities = model.predict_proba(X_test[artifact['feature_columns']])[:, 1]

test_predictions = (test_probabilities >= DECISION_THRESHOLD).astype("int8")

# True Negatives (0 predicted 0)
mask_tn = (y_test == 0) & (test_predictions == 0)
# False Positives (0 predicted 1) - FALSE CALLS
mask_fp = (y_test == 0) & (test_predictions == 1)

X_tn = X_test[mask_tn]
X_fp = X_test[mask_fp]

print(f"True Negatives (TN): {len(X_tn)}")
print(f"False Positives (FP - False Calls): {len(X_fp)}")


True Negatives (TN): 75137
False Positives (FP - False Calls): 1010


## 3. Extract Native XGBoost Tree Thresholds (Per Inspection Type)

In [4]:
# Dump internal trees to DataFrame
booster = model.get_booster()
trees_df = booster.trees_to_dataframe()
explainer = shap.TreeExplainer(model)

print(f"Total internal tree nodes in the model: {len(trees_df)}\n")

unique_types = sorted(X_fp['inspection_type'].unique())

for insp_type in unique_types:
    print("=" * 80)
    print(f"INSPECTION TYPE: {insp_type}")
    print("=" * 80)
    
    # Filter False Positives for this inspection type
    X_fp_type = X_fp[X_fp['inspection_type'] == insp_type]
    print(f"Total False Calls for {insp_type}: {len(X_fp_type)}\n")
    
    if len(X_fp_type) == 0:
        continue
        
    # SHAP Analysis
    try:
        shap_values_fp = explainer.shap_values(X_fp_type)
    except ValueError:
        shap_values_fp = explainer.shap_values(X_fp_type[artifact['feature_columns']])
        
    mean_abs_shap = np.abs(shap_values_fp).mean(axis=0)
    top_feature_indices = np.argsort(mean_abs_shap)[::-1][:10]
    
    # We use the expected feature columns of the model for SHAP indexing
    model_features = artifact['feature_columns'] if 'feature_columns' in artifact else X_fp_type.columns.tolist()
    top_features = [model_features[idx] for idx in top_feature_indices]
    
    print("Top Features Driving False Calls (according to SHAP):")
    for i, feature in enumerate(top_features, 1):
        print(f"  {i}. {feature} (Mean |SHAP|: {mean_abs_shap[top_feature_indices[i-1]]:.4f})")
    
    print("\n-- Native Thresholds for Top Features --")
    for feature in top_features:
        feature_nodes = trees_df[trees_df['Feature'] == feature]
        if len(feature_nodes) == 0:
            continue
            
        threshold_agg = feature_nodes.groupby('Split').agg(
            Frequency=('Split', 'count'),
            Total_Cover=('Cover', 'sum')
        ).sort_values(by='Total_Cover', ascending=False)
        
        top_5 = threshold_agg.head(3)
        thresholds_str = ", ".join([f"< {th:.4f} (Cover: {row['Total_Cover']:.0f})" for th, row in top_5.iterrows()])
        print(f"  * {feature}: {thresholds_str}")
        
    print("\n")


Total internal tree nodes in the model: 5482

INSPECTION TYPE: 1
Total False Calls for 1: 712

Top Features Driving False Calls (according to SHAP):
  1. inspection_feat48 (Mean |SHAP|: 5.5322)
  2. inspection_feat8 (Mean |SHAP|: 5.3484)
  3. inspection_feat1 (Mean |SHAP|: 2.8067)
  4. meta_feat1 (Mean |SHAP|: 1.5426)
  5. inspection_feat95 (Mean |SHAP|: 1.4089)
  6. inspection_feat22 (Mean |SHAP|: 1.3861)
  7. meta_feat4 (Mean |SHAP|: 0.8314)
  8. inspection_feat24 (Mean |SHAP|: 0.5202)
  9. inspection_feat12 (Mean |SHAP|: 0.4759)
  10. inspection_feat28 (Mean |SHAP|: 0.4528)

-- Native Thresholds for Top Features --
  * inspection_feat48: < 0.1167 (Cover: 9263), < 0.7292 (Cover: 4665), < 0.7333 (Cover: 3764)
  * inspection_feat8: < 0.7784 (Cover: 5905), < 0.7676 (Cover: 3190), < 0.7730 (Cover: 2395)
  * inspection_feat1: < 0.3775 (Cover: 1148), < 0.4842 (Cover: 1125), < 0.4388 (Cover: 974)
  * meta_feat1: < 2.0000 (Cover: 8090), < 57.0000 (Cover: 4049), < 53.0000 (Cover: 3325)
  * in

  * inspection_feat95: < 0.3846 (Cover: 9392), < 0.1231 (Cover: 2153), < 0.1385 (Cover: 791)
  * meta_feat4: < 16.0000 (Cover: 2933), < 30.0000 (Cover: 1800), < 6.0000 (Cover: 1761)
  * inspection_feat1: < 0.3775 (Cover: 1148), < 0.4842 (Cover: 1125), < 0.4388 (Cover: 974)
  * inspection_feat28: < 0.5265 (Cover: 3534), < 0.4298 (Cover: 2183), < 0.3320 (Cover: 1714)
  * meta_feat2: < 3.0000 (Cover: 2859), < 2.0000 (Cover: 2358)
  * inspection_feat8: < 0.7784 (Cover: 5905), < 0.7676 (Cover: 3190), < 0.7730 (Cover: 2395)


INSPECTION TYPE: 3
Total False Calls for 3: 210

Top Features Driving False Calls (according to SHAP):
  1. meta_feat1 (Mean |SHAP|: 3.6405)
  2. inspection_feat22 (Mean |SHAP|: 1.3999)
  3. inspection_feat12 (Mean |SHAP|: 1.3143)
  4. inspection_feat95 (Mean |SHAP|: 0.9799)
  5. inspection_feat96 (Mean |SHAP|: 0.7511)
  6. inspection_feat1 (Mean |SHAP|: 0.6415)
  7. meta_feat4 (Mean |SHAP|: 0.6033)
  8. inspection_feat20 (Mean |SHAP|: 0.4614)
  9. inspection_feat11 (Me

  * inspection_feat96: < 0.2673 (Cover: 2203), < 0.3735 (Cover: 1901), < 0.3163 (Cover: 1709)
  * inspection_feat1: < 0.3775 (Cover: 1148), < 0.4842 (Cover: 1125), < 0.4388 (Cover: 974)
  * meta_feat4: < 16.0000 (Cover: 2933), < 30.0000 (Cover: 1800), < 6.0000 (Cover: 1761)
  * inspection_feat20: < 0.4775 (Cover: 862), < 0.4816 (Cover: 781), < 0.4754 (Cover: 679)
  * inspection_feat11: < 0.4239 (Cover: 2199), < 0.6265 (Cover: 1635), < 0.5851 (Cover: 1620)
  * inspection_feat8: < 0.7784 (Cover: 5905), < 0.7676 (Cover: 3190), < 0.7730 (Cover: 2395)


INSPECTION TYPE: 4
Total False Calls for 4: 20

Top Features Driving False Calls (according to SHAP):
  1. inspection_feat22 (Mean |SHAP|: 2.1218)
  2. inspection_feat1 (Mean |SHAP|: 2.0348)
  3. inspection_feat32 (Mean |SHAP|: 1.5606)
  4. inspection_feat8 (Mean |SHAP|: 1.2687)
  5. meta_feat1 (Mean |SHAP|: 1.1360)
  6. meta_feat4 (Mean |SHAP|: 1.0302)
  7. inspection_feat95 (Mean |SHAP|: 0.9546)
  8. inspection_feat3 (Mean |SHAP|: 0.7885)
